# SILVER LAYER Data Cleaning & Transformation

## What is the Silver layer?

The Silver layer is where we clean and purify the raw Bronze data. 

## What this notebook does

1. **Audits** the Bronze data — looks for null values, zero-distance trips, negative fares, etc.
2. **Cleans** the data using a MongoDB Aggregation Pipeline:
   - Removes invalid rows (`trip_distance <= 0`, `fare_amount <= 0`, `passenger_count == 0`)
   - Adds new derived fields: `trip_duration_minutes`, `pickup_hour`, `pickup_day_of_week`, `tip_percentage`
3. **Deduplicates** — finds and removes duplicate rows
4. **Stores** the cleaned data in `silver_trips`

## Why use MongoDB Aggregation Pipeline?

We could clean in Python with pandas, but the **MongoDB aggregation pipeline runs inside the database**. It's much faster on large data because there's no network transfer. 

## Step 1: Connect to MongoDB

In [3]:
import json
import warnings
import pandas as pd
from pymongo import MongoClient
warnings.filterwarnings('ignore')

client = MongoClient('mongodb://localhost:27017/')
db = client['nyc_taxi']

# References to the collections we'll use
bronze_col = db['bronze_trips']    # input: from the bronze layer
silver_col = db['silver_trips']    # output: after cleaning where we are going to save it

bronze_count = bronze_col.count_documents({})
print(f"Connected. Bronze has {bronze_count:,} documents to clean.")

Connected. Bronze has 10,906,858 documents to clean.


## Step 2: Audit the Bronze data find the problems

Before we clean, let's see **what's wrong** with the raw data. We use MongoDB queries to count problematic rows.

In [4]:
print("=" * 60)
print("🔍 DATA QUALITY AUDIT")
print("=" * 60)

total = bronze_col.count_documents({})
print(f"Total Bronze documents: {total:,}\n")

# Count rows with various problems
checks = [
    ('Zero or negative distance',  {'trip_distance': {'$lte': 0}}),
    ('Zero or negative fare',      {'fare_amount': {'$lte': 0}}),
    ('Zero passengers',            {'passenger_count': {'$eq': 0}}),
    ('Negative tip',               {'tip_amount': {'$lt': 0}}),
    ('Missing pickup datetime',    {'tpep_pickup_datetime': {'$eq': None}}),
]

for label, query in checks:
    count = bronze_col.count_documents(query)
    pct = count / total * 100
    print(f"  PROBLEM: {label:<35} {count:>8,}  ({pct:.2f}%)")

print("\nThese are the rows we'll filter out in the Silver layer.")

🔍 DATA QUALITY AUDIT
Total Bronze documents: 10,906,858

  PROBLEM: Zero or negative distance             64,065  (0.59%)
  PROBLEM: Zero or negative fare                  7,393  (0.07%)
  PROBLEM: Zero passengers                          520  (0.00%)
  PROBLEM: Negative tip                             128  (0.00%)
  PROBLEM: Missing pickup datetime                    0  (0.00%)

These are the rows we'll filter out in the Silver layer.


## Step 3: Run the cleaning pipeline

This is the **main logic of the Silver layer**. We use MongoDB's `$aggregate` to:

1. **`$match`**: keep only valid rows (filter out the bad data we found above)
2. **`$addFields`**: compute new useful columns:
   - `trip_duration_minutes` = (dropoff - pickup) in minutes
   - `pickup_hour` = hour of day (0-23) extracted from pickup time
   - `pickup_day_of_week` = 1=Sun, 2=Mon, ..., 7=Sat
   - `tip_percentage` = tip ÷ fare × 100
3. **`$project`**: keep only the columns we need (drop the duplicate raw datetime fields)
4. **`$out`**: write the result to a new collection `silver_trips`

Each step is a **stage** in the pipeline. Documents flow through like water through a series of filters.

In [5]:
# we are dropping the older field so if i run it again it will not give error
silver_col.drop()


pipeline = [
    # STAGE 1: Filter out bad rows
    {'$match': {
        'trip_distance': {'$gt': 0},      # distance must be > 0
        'fare_amount': {'$gt': 0},        # fare must be > 0
        'passenger_count': {'$gt': 0}     # at least 1 passenger
    }},
    
    # STAGE 2: We are adding new columns
    {'$addFields': {
        
        'pickup_datetime': '$tpep_pickup_datetime',
        'dropoff_datetime': '$tpep_dropoff_datetime',
        
        # Trip duration in minutes = (dropoff - pickup) / 60000 ms-per-minute
        'trip_duration_minutes': {
            '$divide': [
                {'$subtract': ['$tpep_dropoff_datetime', '$tpep_pickup_datetime']},
                60000
            ]
        },
        
        # Extract hour of day (0-23) from pickup datetime
        'pickup_hour': {'$hour': '$tpep_pickup_datetime'},
        
        # Extract day of week (1=Sunday, 7=Saturday)
        'pickup_day_of_week': {'$dayOfWeek': '$tpep_pickup_datetime'},
        
        # Tip as % of fare. Use $cond to avoid division by zero.
        'tip_percentage': {
            '$cond': [
                {'$gt': ['$fare_amount', 0]},
                {'$multiply': [{'$divide': ['$tip_amount', '$fare_amount']}, 100]},
                0
            ]
        }
    }},
    
    # STAGE 3: Keep only the fields we want
    {'$project': {
        '_id': 0,                          # drop MongoDB's auto-generated _id
        'VendorID': 1,
        'pickup_datetime': 1,
        'dropoff_datetime': 1,
        'passenger_count': 1,
        'trip_distance': 1,
        'pickup_longitude': 1,
        'pickup_latitude': 1,
        'dropoff_longitude': 1,
        'dropoff_latitude': 1,
        'RatecodeID': 1,
        'payment_type': 1,
        'fare_amount': 1,
        'extra': 1,
        'mta_tax': 1,
        'tip_amount': 1,
        'tolls_amount': 1,
        'improvement_surcharge': 1,
        'total_amount': 1,
        'trip_duration_minutes': 1,
        'pickup_hour': 1,
        'pickup_day_of_week': 1,
        'tip_percentage': 1
    }},
    
    # STAGE 4: Write the result to a new collection
    {'$out': 'silver_trips'}
]

print("⏳ Running aggregation pipeline... (this may take a minute)")
list(bronze_col.aggregate(pipeline))     # execute the pipeline

silver_count = silver_col.count_documents({})
removed = bronze_count - silver_count
print(f"\n Cleaning done!")
print(f"   Bronze: {bronze_count:,} rows")
print(f"   Silver: {silver_count:,} rows")
print(f"   Removed: {removed:,} invalid rows ({removed/bronze_count*100:.2f}%)")

⏳ Running aggregation pipeline... (this may take a minute)

 Cleaning done!
   Bronze: 10,906,858 rows
   Silver: 10,837,367 rows
   Removed: 69,491 invalid rows (0.64%)


## Step 4: Find and remove duplicates

Sometimes the same trip is recorded twice. We define a duplicate as: **same vendor + same pickup time + same dropoff time + same distance + same fare**.

We use `$group` to find groups of identical rows, then delete all but one from each group.

In [6]:
# Find duplicate groups
dup_pipeline = [
    {'$group': {
        '_id': {                                
            'VendorID': '$VendorID',
            'pickup_datetime': '$pickup_datetime',
            'dropoff_datetime': '$dropoff_datetime',
            'trip_distance': '$trip_distance',
            'fare_amount': '$fare_amount'
        },
        'count': {'$sum': 1},                   
        'ids': {'$push': '$_id'}                
    }},
    {'$match': {'count': {'$gt': 1}}}           
]

duplicate_groups = list(silver_col.aggregate(dup_pipeline))
print(f" Found {len(duplicate_groups)} groups of duplicate rows.")

if duplicate_groups:
    # For each group, keep the first ID and delete the rest
    ids_to_delete = []
    for group in duplicate_groups:
        ids_to_delete.extend(group['ids'][1:])    # skip first, delete others
    
    silver_col.delete_many({'_id': {'$in': ids_to_delete}})
    print(f"  Deleted {len(ids_to_delete):,} duplicate documents")
else:
    print(" No duplicates found!")

final_count = silver_col.count_documents({})
print(f"\n Final Silver count: {final_count:,} documents")

 Found 260 groups of duplicate rows.
  Deleted 260 duplicate documents

 Final Silver count: 10,837,107 documents
